# MedicalNLP-RAG — Phase 4B
## Evidence Filtering

Phase 4A established that biomedical evidence improves Qwen3 answer generation, but expert-selected gold evidence still outperformed automatically retrieved Top-10 evidence.

This phase tests whether **reducing retrieval noise** narrows that gap.

---

## Frozen upstream pipeline

`BM25 + MedCPT → RRF → MedCPT Cross-Encoder`

No retrieval model or retrieval parameter is changed in this notebook.

---

## Evidence conditions

### 1. `retrieved_rag`
Phase 4A baseline:
- frozen reranked Top-10 PubMed documents;
- full title + abstract evidence;
- no new generation is performed here: Phase 4A outputs are reused.

### 2. `document_top5`
Simple pruning ablation:
- keep only the first 5 documents from the same frozen ranking;
- full title + abstract;
- tests whether shorter context alone helps.

### 3. `sentence_filtered`
Proposed filtering condition:
- start from the same frozen Top-10 documents;
- split abstracts into sentences;
- score each query–sentence pair with `ncbi/MedCPT-Cross-Encoder`;
- keep the highest-scoring evidence sentences;
- maximum 2 sentences per PubMed document;
- maximum 8 evidence sentences;
- fixed evidence-token budget.

This separates **targeted evidence selection** from simple document truncation.

---

## Same answer generator

Exactly the same Phase 4A LLM and generation policy are retained:

`Qwen/Qwen3-4B-Instruct-2507`

- 4-bit NF4
- greedy decoding
- same JSON schema
- same prompt logic
- no confidence and no abstention yet

---

## Development selection score

For internal **validation-only method selection**, define:

`MacroTaskScore = mean(YesNo Accuracy, Factoid MRR, List F1, Summary ROUGE-L)`

Each BioASQ question type receives equal weight.

This is an **internal development selection score**, not an official BioASQ metric.

Primary selection rule:
1. highest MacroTaskScore;
2. tie-break: higher overall ROUGE-L.

The 860-question internal test remains locked.

## 0. Configuration

In [ ]:
PROJECT_NAME = "MedicalNLP_RAG"
SEED = 42

# Same answer generator as Phase 4A
LLM_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
USE_4BIT = True
LLM_BATCH_SIZE = 2

# Frozen retrieval input
SOURCE_TOP_K = 10

# Evidence filtering
DOCUMENT_TOP_K = 5

SENTENCE_MAX_PER_DOCUMENT = 2
SENTENCE_MAX_TOTAL = 8
SENTENCE_MIN_CHARS = 25
SENTENCE_FILTER_TOKEN_BUDGET = 1800

# Sentence scoring
CROSS_ENCODER_MODEL = "ncbi/MedCPT-Cross-Encoder"
CROSS_ENCODER_MAX_LENGTH = 512
CROSS_ENCODER_BATCH_SIZE = 48

# Generation
MAX_INPUT_TOKENS = 8192
MAX_NEW_TOKENS = 320

NEW_CONDITIONS = [
    "document_top5",
    "sentence_filtered"
]

COMPARISON_CONDITIONS = [
    "retrieved_rag",
    "document_top5",
    "sentence_filtered"
]

# First run: smoke only.
RUN_SMOKE_TEST = True
RUN_FULL_GENERATION = False
SMOKE_PER_TYPE = 3

ALLOW_INTERNAL_TEST_EVALUATION = False

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Dependency check

In [ ]:
import sys
import subprocess
import importlib
import importlib.util
from packaging.version import Version

def get_version(name):
    try:
        return getattr(importlib.import_module(name), "__version__", "0")
    except Exception:
        return None

tv = get_version("transformers")

need_install = (
    tv is None
    or Version(tv) < Version("4.51.0")
    or importlib.util.find_spec("bitsandbytes") is None
    or importlib.util.find_spec("accelerate") is None
    or importlib.util.find_spec("rouge_score") is None
)

if need_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.51.0",
        "accelerate>=1.0.0",
        "bitsandbytes>=0.45.0",
        "rouge-score>=0.1.2",
    ])

print("Dependency check complete.")

## 3. Imports and project paths

In [ ]:
from pathlib import Path
import os
import re
import gc
import sys
import json
import math
import time
import random
import hashlib
import unicodedata
import platform
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from sklearn.metrics import f1_score
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path(f"/content/drive/MyDrive/{PROJECT_NAME}")

SPLIT_DIR = ROOT / "data" / "bioasq14_splits"
CACHE_DIR = ROOT / "data" / "pubmed_cache"

PHASE3C_DIR = ROOT / "results" / "phase3C_reranking"
PHASE3C_TABLE_DIR = PHASE3C_DIR / "tables"
PHASE3C_SCORE_DIR = PHASE3C_DIR / "score_cache"

PHASE4A_DIR = ROOT / "results" / "phase4A_answer_generation"
PHASE4A_GEN_DIR = PHASE4A_DIR / "generations"

PHASE4B_DIR = ROOT / "results" / "phase4B_evidence_filtering"
TABLE_DIR = PHASE4B_DIR / "tables"
FIG_DIR = PHASE4B_DIR / "figures"
GEN_DIR = PHASE4B_DIR / "generations"
FILTER_CACHE_DIR = PHASE4B_DIR / "filter_cache"
LOG_DIR = PHASE4B_DIR / "logs"

for p in [
    PHASE4B_DIR,
    TABLE_DIR,
    FIG_DIR,
    GEN_DIR,
    FILTER_CACHE_DIR,
    LOG_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

VALIDATION_JSON = SPLIT_DIR / "validation.json"
CORPUS_CSV = CACHE_DIR / "pubmed_closed_corpus.csv"
FROZEN_RUN = PHASE3C_TABLE_DIR / "RRF_MedCPT_CrossEncoder_top100.jsonl"
FROZEN_SELECTION = PHASE3C_DIR / "frozen_retrieval_pipeline.json"
DOC_SCORE_CACHE = PHASE3C_SCORE_DIR / "medcpt_crossencoder_scores.csv"

PHASE4A_GENERATIONS = (
    PHASE4A_GEN_DIR / "validation_862q_3conditions.jsonl"
)

assert VALIDATION_JSON.exists()
assert CORPUS_CSV.exists()
assert FROZEN_RUN.exists()
assert FROZEN_SELECTION.exists()
assert DOC_SCORE_CACHE.exists()
assert PHASE4A_GENERATIONS.exists()

print("Phase 4B output:", PHASE4B_DIR)

## 4. Locked-test safety

In [ ]:
if ALLOW_INTERNAL_TEST_EVALUATION:
    raise RuntimeError(
        "Internal test must remain locked during Phase 4B."
    )

print("INTERNAL TEST REMAINS LOCKED.")

## 5. Verify frozen retrieval pipeline

In [ ]:
with open(FROZEN_SELECTION, "r", encoding="utf-8") as f:
    frozen_selection = json.load(f)

assert frozen_selection["frozen_pipeline"] == "RRF+MedCPT_CrossEncoder"

print(json.dumps(frozen_selection, indent=2))

## 6. Load validation questions

In [ ]:
with open(VALIDATION_JSON, "r", encoding="utf-8") as f:
    validation_questions = json.load(f)["questions"]

assert len(validation_questions) == 862

rows = []

for q in validation_questions:
    rows.append({
        "id": str(q["id"]),
        "type": str(q["type"]).strip().lower(),
        "body": str(q["body"]).strip(),
        "exact_answer_gold": q.get("exact_answer"),
        "ideal_answer_gold": q.get("ideal_answer"),
        "snippets_gold": q.get("snippets", []),
        "documents_gold": q.get("documents", []),
    })

val_df = pd.DataFrame(rows)

assert val_df["id"].nunique() == 862
assert set(val_df["type"]) == {"yesno", "factoid", "list", "summary"}

question_lookup = {
    str(q["id"]): q
    for q in validation_questions
}

display(val_df["type"].value_counts())

## 7. Load corpus and frozen rankings

In [ ]:
corpus_df = pd.read_csv(
    CORPUS_CSV,
    dtype={"pmid": str}
)

corpus_df["pmid"] = (
    corpus_df["pmid"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

corpus_df["title"] = (
    corpus_df["title"]
    .fillna("")
    .astype(str)
    .str.strip()
)

corpus_df["abstract"] = (
    corpus_df["abstract"]
    .fillna("")
    .astype(str)
    .str.strip()
)

corpus_lookup = (
    corpus_df
    .drop_duplicates("pmid")
    .set_index("pmid")[["title", "abstract"]]
    .to_dict(orient="index")
)

frozen_rankings = {}

with open(FROZEN_RUN, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        frozen_rankings[str(rec["id"])] = [
            str(x)
            for x in rec["ranked_pmids"]
        ]

assert len(frozen_rankings) == 862
assert all(len(x) >= SOURCE_TOP_K for x in frozen_rankings.values())

print("Corpus records:", len(corpus_lookup))
print("Frozen rankings:", len(frozen_rankings))

## 8. Load Phase 3C document Cross-Encoder scores

In [ ]:
doc_scores = pd.read_csv(
    DOC_SCORE_CACHE,
    dtype={"id": str, "pmid": str}
)

doc_scores = (
    doc_scores
    .drop_duplicates(["id", "pmid"], keep="last")
    .reset_index(drop=True)
)

doc_score_lookup = {
    (str(r.id), str(r.pmid)): float(r.cross_score)
    for r in doc_scores.itertuples(index=False)
}

print("Document Cross-Encoder scores:", len(doc_score_lookup))

## 9. Load Phase 4A retrieved-RAG baseline outputs

In [ ]:
phase4a_records = []

with open(PHASE4A_GENERATIONS, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            phase4a_records.append(json.loads(line))

phase4a_df = pd.DataFrame(phase4a_records)

baseline_rag = (
    phase4a_df[
        phase4a_df["condition"] == "retrieved_rag"
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(baseline_rag) == 862
assert baseline_rag["id"].astype(str).nunique() == 862

print("Loaded Phase 4A retrieved-RAG baseline:", len(baseline_rag))
print("Valid JSON rate:", baseline_rag["parse_valid"].mean())

# Part A — Build filtering candidates

Sentence splitting is intentionally deterministic and dependency-free.

The splitter is conservative:
- it primarily splits after `.`, `?`, or `!`;
- very short fragments are discarded;
- exact normalized duplicate sentences are removed within each query.

## 10. Deterministic sentence splitter

In [ ]:
def normalize_sentence_key(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def split_sentences(text):
    text = " ".join(str(text).split()).strip()

    if not text:
        return []

    # Conservative biomedical sentence split.
    parts = re.split(
        r"(?<=[.!?])\s+(?=[A-Z0-9])",
        text
    )

    out = []

    for part in parts:
        part = part.strip()

        if len(part) >= SENTENCE_MIN_CHARS:
            out.append(part)

    return out

## 11. Build query–sentence candidate table from frozen Top-10

In [ ]:
sentence_rows = []

for row in tqdm(
    val_df.itertuples(index=False),
    total=len(val_df),
    desc="Building sentence candidates"
):
    seen = set()

    for document_rank, pmid in enumerate(
        frozen_rankings[row.id][:SOURCE_TOP_K],
        start=1
    ):
        rec = corpus_lookup.get(pmid)

        if rec is None:
            continue

        title = rec["title"]
        abstract = rec["abstract"]

        sentences = split_sentences(abstract)

        # If an abstract could not be split but contains useful text,
        # keep it as one candidate.
        if not sentences and len(abstract) >= SENTENCE_MIN_CHARS:
            sentences = [abstract]

        for sentence_index, sentence in enumerate(
            sentences,
            start=1
        ):
            key = normalize_sentence_key(sentence)

            if not key or key in seen:
                continue

            seen.add(key)

            candidate_text = (
                f"{title}. {sentence}"
                if title
                else sentence
            )

            sentence_rows.append({
                "id": row.id,
                "type": row.type,
                "question": row.body,
                "pmid": pmid,
                "document_rank": document_rank,
                "document_cross_score": doc_score_lookup.get(
                    (row.id, pmid),
                    np.nan
                ),
                "sentence_index": sentence_index,
                "title": title,
                "sentence": sentence,
                "candidate_text": candidate_text,
            })

sentence_candidates = pd.DataFrame(sentence_rows)

print("Sentence candidates:", len(sentence_candidates))
print(
    "Mean candidates/query:",
    round(len(sentence_candidates) / len(val_df), 2)
)

display(
    sentence_candidates.groupby("type")
    .size()
    .rename("n_sentence_candidates")
)

# Part B — Sentence-level MedCPT scoring

The official MedCPT Cross-Encoder documentation states that **higher scores indicate higher relevance**.  
We use the same model zero-shot at sentence level as an evidence selector.

The score is used for ranking only; it is **not treated as calibrated probability**.

## 12. GPU check

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a Colab GPU before running Phase 4B."
    )

DEVICE = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))

## 13. Load MedCPT Cross-Encoder

In [ ]:
cross_tokenizer = AutoTokenizer.from_pretrained(
    CROSS_ENCODER_MODEL
)

cross_model = AutoModelForSequenceClassification.from_pretrained(
    CROSS_ENCODER_MODEL
).to(DEVICE)

cross_model.eval()

print("Loaded:", CROSS_ENCODER_MODEL)

## 14. Resumable sentence scoring

In [ ]:
SENTENCE_SCORE_CACHE = (
    FILTER_CACHE_DIR / "sentence_crossencoder_scores.csv"
)

def sentence_key(row):
    raw = (
        str(row["id"])
        + "\t"
        + str(row["pmid"])
        + "\t"
        + str(row["sentence_index"])
        + "\t"
        + str(row["sentence"])
    )

    return hashlib.sha256(
        raw.encode("utf-8")
    ).hexdigest()

sentence_candidates["sentence_key"] = sentence_candidates.apply(
    sentence_key,
    axis=1
)

if SENTENCE_SCORE_CACHE.exists():
    scored_existing = pd.read_csv(
        SENTENCE_SCORE_CACHE,
        dtype={"sentence_key": str}
    )

    scored_existing = scored_existing.drop_duplicates(
        "sentence_key",
        keep="last"
    )

    completed_keys = set(
        scored_existing["sentence_key"]
    )
else:
    completed_keys = set()

todo = sentence_candidates[
    ~sentence_candidates["sentence_key"].isin(completed_keys)
].reset_index(drop=True)

print("Already scored:", len(completed_keys))
print("Remaining:", len(todo))

In [ ]:
buffer = []

def flush_sentence_score_buffer(buffer):
    if not buffer:
        return []

    chunk = pd.concat(
        buffer,
        ignore_index=True
    )

    if SENTENCE_SCORE_CACHE.exists():
        chunk.to_csv(
            SENTENCE_SCORE_CACHE,
            mode="a",
            header=False,
            index=False
        )
    else:
        chunk.to_csv(
            SENTENCE_SCORE_CACHE,
            index=False
        )

    return []

for start in tqdm(
    range(0, len(todo), CROSS_ENCODER_BATCH_SIZE),
    desc="Sentence-level MedCPT filtering"
):
    batch = todo.iloc[
        start:start + CROSS_ENCODER_BATCH_SIZE
    ]

    pairs = [
        [q, a]
        for q, a in zip(
            batch["question"],
            batch["candidate_text"]
        )
    ]

    enc = cross_tokenizer(
        pairs,
        truncation=True,
        padding=True,
        return_tensors="pt",
        max_length=CROSS_ENCODER_MAX_LENGTH
    )

    enc = {
        k: v.to(DEVICE)
        for k, v in enc.items()
    }

    with torch.no_grad():
        logits = cross_model(**enc).logits

        if logits.ndim == 2 and logits.shape[1] == 1:
            scores = logits[:, 0]
        else:
            scores = logits.squeeze()

        scores = (
            scores
            .detach()
            .float()
            .cpu()
            .numpy()
        )

    buffer.append(
        pd.DataFrame({
            "sentence_key": batch["sentence_key"].values,
            "sentence_cross_score": scores
        })
    )

    if len(buffer) >= 40:
        buffer = flush_sentence_score_buffer(buffer)

buffer = flush_sentence_score_buffer(buffer)

print("Sentence scoring pass complete.")

## 15. Validate sentence score cache

In [ ]:
sentence_scores = pd.read_csv(
    SENTENCE_SCORE_CACHE,
    dtype={"sentence_key": str}
)

sentence_scores = (
    sentence_scores
    .drop_duplicates("sentence_key", keep="last")
    .reset_index(drop=True)
)

assert len(sentence_scores) == len(sentence_candidates)
assert sentence_scores["sentence_cross_score"].notna().all()

sentence_candidates = sentence_candidates.merge(
    sentence_scores,
    on="sentence_key",
    validate="one_to_one"
)

display(
    sentence_candidates["sentence_cross_score"]
    .describe()
)

## 16. Unload Cross-Encoder before loading Qwen

This releases GPU memory.

In [ ]:
del cross_model
del cross_tokenizer

gc.collect()
torch.cuda.empty_cache()

print("Cross-Encoder unloaded.")

# Part C — Construct filtered evidence

## Document Top-5
Keeps the first five frozen reranked documents.

## Sentence-filtered evidence
Selection is deterministic:

1. sort candidate sentences by sentence Cross-Encoder score descending;
2. maximum 2 selected sentences from any PMID;
3. maximum 8 sentences total;
4. apply a fixed 1,800-token evidence budget.

No score threshold is used because raw Cross-Encoder logits are not assumed to be calibrated probabilities.

## 17. Load Qwen tokenizer for token-budget accounting

In [ ]:
llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL,
    use_fast=True
)

if llm_tokenizer.pad_token_id is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

llm_tokenizer.padding_side = "left"

print("Tokenizer loaded:", LLM_MODEL)

## 18. Token helpers

In [ ]:
def llm_token_count(text):
    return len(
        llm_tokenizer(
            text,
            add_special_tokens=False
        )["input_ids"]
    )


def llm_truncate(text, max_tokens):
    ids = llm_tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    if len(ids) <= max_tokens:
        return text

    return llm_tokenizer.decode(
        ids[:max_tokens],
        skip_special_tokens=True
    )


def pack_to_budget(blocks, max_tokens):
    packed = []
    used = 0

    for block in blocks:
        block = str(block).strip()

        if not block:
            continue

        n = llm_token_count(block)

        if used + n <= max_tokens:
            packed.append(block)
            used += n

        else:
            remaining = max_tokens - used

            if remaining > 32:
                packed.append(
                    llm_truncate(block, remaining)
                )
                used = max_tokens

            break

    return "\n\n".join(packed), used


print("Token helper functions ready.")

## 19. Document Top-5 evidence

In [ ]:
def build_document_top5_evidence(qid):
    blocks = []

    for rank, pmid in enumerate(
        frozen_rankings[qid][:DOCUMENT_TOP_K],
        start=1
    ):
        rec = corpus_lookup.get(pmid)

        if rec is None:
            continue

        blocks.append(
            f"[Filtered document {rank}] PMID: {pmid}\n"
            f"Title: {rec['title']}\n"
            f"Abstract: {rec['abstract']}"
        )

    # Keep the same generous ceiling as Phase 4A.
    # Actual Top-5 context is usually much shorter.
    return pack_to_budget(
        blocks,
        max_tokens=6000
    )

## 20. Sentence-filtered evidence

In [ ]:
sentence_groups = {
    qid: group.copy()
    for qid, group in sentence_candidates.groupby(
        "id",
        sort=False
    )
}

def select_sentence_evidence(qid):
    group = sentence_groups[qid].sort_values(
        [
            "sentence_cross_score",
            "document_rank",
            "sentence_index",
        ],
        ascending=[False, True, True]
    )

    selected = []
    per_doc_counts = {}

    for row in group.itertuples(index=False):
        current = per_doc_counts.get(
            row.pmid,
            0
        )

        if current >= SENTENCE_MAX_PER_DOCUMENT:
            continue

        selected.append(row)
        per_doc_counts[row.pmid] = current + 1

        if len(selected) >= SENTENCE_MAX_TOTAL:
            break

    blocks = []

    for evidence_rank, row in enumerate(
        selected,
        start=1
    ):
        blocks.append(
            f"[Filtered evidence {evidence_rank}] PMID: {row.pmid}\n"
            f"Title: {row.title}\n"
            f"Evidence sentence: {row.sentence}"
        )

    packed, token_count = pack_to_budget(
        blocks,
        max_tokens=SENTENCE_FILTER_TOKEN_BUDGET
    )

    selected_meta = [
        {
            "pmid": str(r.pmid),
            "sentence_cross_score": float(r.sentence_cross_score),
            "document_rank": int(r.document_rank),
            "sentence_index": int(r.sentence_index),
            "sentence": str(r.sentence),
        }
        for r in selected
    ]

    return packed, token_count, selected_meta

## 21. Build and freeze Phase 4B evidence cache

In [ ]:
EVIDENCE_CACHE_JSONL = (
    FILTER_CACHE_DIR / "phase4B_evidence_sets.jsonl"
)

evidence_rows = []

for row in tqdm(
    val_df.itertuples(index=False),
    total=len(val_df),
    desc="Building filtered evidence"
):
    top5_text, top5_tokens = build_document_top5_evidence(
        row.id
    )

    sentence_text, sentence_tokens, selected_meta = (
        select_sentence_evidence(row.id)
    )

    evidence_rows.append({
        "id": row.id,
        "type": row.type,
        "document_top5_text": top5_text,
        "document_top5_tokens": int(top5_tokens),
        "sentence_filtered_text": sentence_text,
        "sentence_filtered_tokens": int(sentence_tokens),
        "sentence_selected": selected_meta,
    })

with open(
    EVIDENCE_CACHE_JSONL,
    "w",
    encoding="utf-8"
) as f:
    for rec in evidence_rows:
        f.write(
            json.dumps(
                rec,
                ensure_ascii=False
            )
            + "\n"
        )

evidence_df = pd.DataFrame(evidence_rows)

assert len(evidence_df) == 862

print("Frozen evidence cache:", EVIDENCE_CACHE_JSONL)

In [ ]:
print("pack_to_budget:", callable(pack_to_budget))
print("build_document_top5_evidence:", callable(build_document_top5_evidence))
print("select_sentence_evidence:", callable(select_sentence_evidence))

## 22. Evidence-length comparison

In [ ]:
# Phase 4A retrieved evidence lengths are available from its outputs.
baseline_token_df = (
    baseline_rag[
        ["id", "type", "evidence_tokens"]
    ]
    .rename(
        columns={
            "evidence_tokens":
            "retrieved_rag_tokens"
        }
    )
)

length_df = (
    evidence_df[
        [
            "id",
            "type",
            "document_top5_tokens",
            "sentence_filtered_tokens",
        ]
    ]
    .merge(
        baseline_token_df,
        on=["id", "type"],
        validate="one_to_one"
    )
)

length_summary = (
    length_df.groupby("type")[
        [
            "retrieved_rag_tokens",
            "document_top5_tokens",
            "sentence_filtered_tokens",
        ]
    ]
    .agg(["mean", "median", "std", "min", "max"])
    .round(1)
)

length_df.to_csv(
    TABLE_DIR / "evidence_length_per_question.csv",
    index=False
)

length_summary.to_csv(
    TABLE_DIR / "evidence_length_summary.csv"
)

display(length_summary)

## 23. Gold-document coverage after filtering

In [ ]:
def extract_pmid(ref):
    s = str(ref).strip().rstrip("/")
    m = re.search(r"(\d+)(?:\D*)$", s)
    return m.group(1) if m else None

gold_pmids = {}

for q in validation_questions:
    gold = {
        extract_pmid(ref)
        for ref in q.get("documents", [])
    }
    gold.discard(None)
    gold_pmids[str(q["id"])] = gold

coverage_rows = []

for row in val_df.itertuples(index=False):
    gold = gold_pmids[row.id]

    rag_pmids = set(
        frozen_rankings[row.id][:SOURCE_TOP_K]
    )

    top5_pmids = set(
        frozen_rankings[row.id][:DOCUMENT_TOP_K]
    )

    sentence_meta = evidence_df.loc[
        evidence_df["id"] == row.id,
        "sentence_selected"
    ].iloc[0]

    sentence_pmids = {
        str(x["pmid"])
        for x in sentence_meta
    }

    coverage_rows.append({
        "id": row.id,
        "type": row.type,
        "n_gold": len(gold),
        "retrieved_rag_doc_recall": (
            len(gold & rag_pmids) / len(gold)
            if gold else np.nan
        ),
        "document_top5_doc_recall": (
            len(gold & top5_pmids) / len(gold)
            if gold else np.nan
        ),
        "sentence_filtered_doc_recall": (
            len(gold & sentence_pmids) / len(gold)
            if gold else np.nan
        ),
        "sentence_filtered_unique_docs": len(
            sentence_pmids
        ),
    })

coverage_df = pd.DataFrame(
    coverage_rows
)

coverage_df.to_csv(
    TABLE_DIR / "filter_document_coverage_per_question.csv",
    index=False
)

coverage_summary = (
    coverage_df.groupby("type")[
        [
            "retrieved_rag_doc_recall",
            "document_top5_doc_recall",
            "sentence_filtered_doc_recall",
            "sentence_filtered_unique_docs",
        ]
    ]
    .mean()
    .round(4)
)

display(coverage_summary)

# Part D — Same Phase 4A answer prompt

## 24. Fixed prompt

In [ ]:
SYSTEM_PROMPT = """You are a biomedical question-answering model used in a research benchmark.

Rules:
1. Return JSON only. Do not use Markdown or code fences.
2. Do not provide hidden reasoning or chain-of-thought.
3. Follow the required output shape exactly.
4. The ideal_answer must be concise, factual, and at most 150 words.
5. Do not invent citations or PMID identifiers.
"""

FORMAT_RULES = {
    "yesno": """Question type: yesno.
Return exactly this JSON structure:
{"exact_answer":"yes","ideal_answer":"concise biomedical explanation"}
or use "no" as exact_answer.
The exact_answer must be exactly yes or no.""",

    "factoid": """Question type: factoid.
Return exactly this JSON structure:
{"exact_answer":["answer 1","answer 2"],"ideal_answer":"concise biomedical explanation"}
The exact_answer must contain at most 5 short candidate answers, ordered from most to least likely.""",

    "list": """Question type: list.
Return exactly this JSON structure:
{"exact_answer":["item 1","item 2"],"ideal_answer":"concise biomedical explanation"}
The exact_answer must be a JSON list of distinct short items.""",

    "summary": """Question type: summary.
Return exactly this JSON structure:
{"exact_answer":[],"ideal_answer":"concise biomedical summary"}
The exact_answer must be an empty JSON list."""
}

def build_messages(question, qtype, condition, evidence):
    if condition == "document_top5":
        evidence_rule = (
            "Automatically retrieved PubMed evidence is provided below after "
            "document-level pruning. Base the answer on this evidence and do "
            "not add unsupported biomedical claims."
        )
    elif condition == "sentence_filtered":
        evidence_rule = (
            "Automatically retrieved PubMed evidence has been filtered to "
            "high-relevance evidence sentences. Base the answer on this evidence "
            "and do not add unsupported biomedical claims."
        )
    else:
        raise ValueError(condition)

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": "\\n\\n".join([
                FORMAT_RULES[qtype],
                evidence_rule,
                f"Question: {question}",
                "Evidence:\\n" + evidence,
            ]),
        },
    ]

## 25. JSON parser

In [ ]:
def clean_model_text(text):
    text = str(text).strip()

    if text.startswith("```"):
        text = re.sub(
            r"^```(?:json)?\\s*",
            "",
            text,
            flags=re.IGNORECASE
        )
        text = re.sub(
            r"\\s*```$",
            "",
            text
        )

    return text.strip()

def extract_json_object(text):
    text = clean_model_text(text)

    try:
        return json.loads(text), "direct"
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start >= 0 and end > start:
        try:
            return json.loads(
                text[start:end + 1]
            ), "extracted"
        except Exception:
            pass

    return None, "invalid"

def validate_prediction_object(obj, qtype):
    if not isinstance(obj, dict):
        return False, "not_dict"

    if (
        "exact_answer" not in obj
        or "ideal_answer" not in obj
    ):
        return False, "missing_required_field"

    if not isinstance(
        obj["ideal_answer"],
        str
    ):
        return False, "ideal_not_string"

    exact = obj["exact_answer"]

    if qtype == "yesno":
        if not isinstance(exact, str):
            return False, "yesno_exact_not_string"

        if exact.strip().lower() not in {
            "yes",
            "no"
        }:
            return False, "yesno_invalid_value"

    elif qtype in {"factoid", "list"}:
        if not isinstance(exact, list):
            return False, "exact_not_list"

        if not all(
            isinstance(x, str)
            for x in exact
        ):
            return False, "exact_list_nonstring"

        if (
            qtype == "factoid"
            and len(exact) > 5
        ):
            return False, "factoid_more_than_5"

    elif qtype == "summary":
        if (
            not isinstance(exact, list)
            or len(exact) != 0
        ):
            return False, "summary_exact_not_empty"

    return True, "valid"

def parse_prediction(raw_text, qtype):
    obj, parse_mode = extract_json_object(
        raw_text
    )

    if obj is None:
        return {
            "parse_valid": False,
            "parse_mode": parse_mode,
            "validation_status": "invalid_json",
            "exact_answer_pred": None,
            "ideal_answer_pred": "",
        }

    valid, status = validate_prediction_object(
        obj,
        qtype
    )

    return {
        "parse_valid": bool(valid),
        "parse_mode": parse_mode,
        "validation_status": status,
        "exact_answer_pred": obj.get(
            "exact_answer"
        ),
        "ideal_answer_pred": (
            obj.get("ideal_answer", "")
            if isinstance(
                obj.get("ideal_answer", ""),
                str
            )
            else ""
        ),
    }

# Part E — Load Qwen and generate new filtering conditions

## 26. Load Qwen

In [ ]:
if USE_4BIT:
    qconfig = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        quantization_config=qconfig,
        device_map="auto",
        torch_dtype=torch.float16,
    )
else:
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        device_map="auto",
        torch_dtype="auto",
    )

llm_model.eval()

llm_model.generation_config.pad_token_id = (
    llm_tokenizer.pad_token_id
)
llm_model.generation_config.eos_token_id = (
    llm_tokenizer.eos_token_id
)

print("Loaded:", LLM_MODEL)

## 27. Evidence lookup

In [ ]:
evidence_lookup = (
    evidence_df
    .set_index("id")
    .to_dict(orient="index")
)

def get_filtered_evidence(qid, condition):
    rec = evidence_lookup[str(qid)]

    if condition == "document_top5":
        return (
            rec["document_top5_text"],
            int(rec["document_top5_tokens"])
        )

    if condition == "sentence_filtered":
        return (
            rec["sentence_filtered_text"],
            int(rec["sentence_filtered_tokens"])
        )

    raise ValueError(condition)

## 28. Prepare generation job

In [ ]:
def prepare_generation_row(row, condition):
    evidence, evidence_tokens = (
        get_filtered_evidence(
            row["id"],
            condition
        )
    )

    messages = build_messages(
        question=row["body"],
        qtype=row["type"],
        condition=condition,
        evidence=evidence,
    )

    prompt_text = (
        llm_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    )

    return {
        "id": str(row["id"]),
        "type": row["type"],
        "question": row["body"],
        "condition": condition,
        "evidence_tokens": int(
            evidence_tokens
        ),
        "prompt_text": prompt_text,
    }

## 29. Resumable deterministic generation

In [ ]:
def generation_path(run_name):
    return GEN_DIR / f"{run_name}.jsonl"

def load_existing_generations(path):
    out = {}

    if not path.exists():
        return out

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        for line in f:
            if not line.strip():
                continue

            rec = json.loads(line)

            out[
                (
                    str(rec["id"]),
                    str(rec["condition"])
                )
            ] = rec

    return out

def append_jsonl(path, records):
    with open(
        path,
        "a",
        encoding="utf-8"
    ) as f:
        for rec in records:
            f.write(
                json.dumps(
                    rec,
                    ensure_ascii=False,
                    default=str
                )
                + "\n"
            )

        f.flush()

def generate_conditions(
    input_df,
    run_name
):
    path = generation_path(
        run_name
    )

    existing = (
        load_existing_generations(
            path
        )
    )

    jobs = []

    for _, row in input_df.iterrows():
        for condition in NEW_CONDITIONS:
            key = (
                str(row["id"]),
                condition
            )

            if key not in existing:
                jobs.append(
                    prepare_generation_row(
                        row,
                        condition
                    )
                )

    print("Run:", run_name)
    print("Existing:", len(existing))
    print("Remaining:", len(jobs))

    for start in tqdm(
        range(0, len(jobs), LLM_BATCH_SIZE),
        desc=run_name
    ):
        batch_jobs = jobs[
            start:start + LLM_BATCH_SIZE
        ]

        prompts = [
            x["prompt_text"]
            for x in batch_jobs
        ]

        inputs = llm_tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        )

        inputs = {
            k: v.to(llm_model.device)
            for k, v in inputs.items()
        }

        input_width = (
            inputs["input_ids"].shape[1]
        )

        with torch.inference_mode():
            output_ids = llm_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
            )

        generated_only = (
            output_ids[:, input_width:]
        )

        decoded = (
            llm_tokenizer.batch_decode(
                generated_only,
                skip_special_tokens=True
            )
        )

        records = []

        for job, raw in zip(
            batch_jobs,
            decoded
        ):
            parsed = parse_prediction(
                raw,
                job["type"]
            )

            records.append({
                "id": job["id"],
                "type": job["type"],
                "condition": job["condition"],
                "question": job["question"],
                "evidence_tokens": job[
                    "evidence_tokens"
                ],
                "raw_output": raw,
                **parsed,
            })

        append_jsonl(
            path,
            records
        )

    complete = (
        load_existing_generations(
            path
        )
    )

    requested_ids = set(
        input_df["id"].astype(str)
    )

    return pd.DataFrame([
        rec
        for rec in complete.values()
        if str(rec["id"])
        in requested_ids
    ])

# Part F — 12-question smoke test

The same deterministic 12-question stratified sample is used.

New generations:

`12 × 2 = 24`

## 30. Build smoke sample

In [ ]:
smoke_parts = []

for qtype in sorted(
    val_df["type"].unique()
):
    smoke_parts.append(
        val_df[
            val_df["type"] == qtype
        ].sample(
            n=SMOKE_PER_TYPE,
            random_state=SEED
        )
    )

smoke_df = (
    pd.concat(
        smoke_parts,
        ignore_index=True
    )
    .sort_values(["type", "id"])
    .reset_index(drop=True)
)

display(
    smoke_df[
        ["id", "type", "body"]
    ]
)

## 31. Run smoke test

In [ ]:
smoke_results = None

if RUN_SMOKE_TEST:
    smoke_results = generate_conditions(
        smoke_df,
        "smoke_12q_2filtered_conditions"
    )

    assert len(smoke_results) == 24

    smoke_results.to_csv(
        TABLE_DIR
        / "smoke_12q_2filtered_conditions.csv",
        index=False
    )

    display(
        smoke_results.groupby(
            ["condition", "type"]
        )["parse_valid"]
        .agg(["count", "mean"])
    )
else:
    print("Smoke test skipped.")

## 32. Inspect smoke outputs

In [ ]:
if smoke_results is not None:
    display(
        smoke_results[
            [
                "id",
                "type",
                "condition",
                "evidence_tokens",
                "parse_valid",
                "validation_status",
                "exact_answer_pred",
                "ideal_answer_pred",
            ]
        ].sort_values(
            ["type", "id", "condition"]
        )
    )

# STOP POINT 4B-SMOKE

Stop here first.

Check:
- 24/24 or near-24/24 valid JSON;
- document Top-5 and sentence-filtered evidence lengths;
- no empty sentence-filtered evidence;
- correct exact-answer shapes.

Keep:

```python
RUN_FULL_GENERATION = False
```

until the smoke output is reviewed.

# Part G — Evaluation

Invalid JSON outputs are handled conservatively:
- exact-answer metric = **0**
- ROUGE = **0**

They are **not dropped** from denominators or bootstrap comparisons.

## 33. Normalization and gold helpers

In [ ]:
def normalize_answer_text(text):
    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )
    text = text.lower().strip()
    text = re.sub(r"[_]+", " ", text)
    text = re.sub(
        r"[^\w\s\.\-\+/]",
        " ",
        text
    )
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()
    return text

def as_gold_groups(
    exact_answer,
    qtype
):
    if qtype == "yesno":
        if exact_answer is None:
            return []

        return [[
            normalize_answer_text(
                exact_answer
            )
        ]]

    if qtype not in {
        "factoid",
        "list"
    }:
        return []

    if exact_answer is None:
        return []

    if not isinstance(
        exact_answer,
        list
    ):
        exact_answer = [
            exact_answer
        ]

    groups = []

    for item in exact_answer:
        if isinstance(item, list):
            syns = [
                normalize_answer_text(x)
                for x in item
                if str(x).strip()
            ]
        else:
            syns = [
                normalize_answer_text(
                    item
                )
            ]

        syns = [
            x for x in syns
            if x
        ]

        if syns:
            groups.append(syns)

    return groups

def prediction_list(pred):
    if isinstance(pred, list):
        return [
            str(x).strip()
            for x in pred
            if str(x).strip()
        ]

    if (
        isinstance(pred, str)
        and pred.strip()
    ):
        return [pred.strip()]

    return []

## 34. Exact-answer scoring

In [ ]:
def score_factoid(pred, gold):
    preds = prediction_list(
        pred
    )[:5]

    groups = as_gold_groups(
        gold,
        "factoid"
    )

    all_gold = {
        syn
        for group in groups
        for syn in group
    }

    norm_preds = [
        normalize_answer_text(x)
        for x in preds
    ]

    strict = float(
        bool(norm_preds)
        and norm_preds[0] in all_gold
    )

    first_rank = None

    for rank, p in enumerate(
        norm_preds,
        start=1
    ):
        if p in all_gold:
            first_rank = rank
            break

    return {
        "factoid_strict": strict,
        "factoid_lenient": float(
            first_rank is not None
        ),
        "factoid_mrr": (
            1.0 / first_rank
            if first_rank
            else 0.0
        ),
    }

def score_list(pred, gold):
    preds = prediction_list(pred)

    groups = as_gold_groups(
        gold,
        "list"
    )

    unmatched = set(
        range(len(groups))
    )

    matched = 0

    for pred_item in preds:
        p = normalize_answer_text(
            pred_item
        )

        for gi in list(
            unmatched
        ):
            if p in set(
                groups[gi]
            ):
                matched += 1
                unmatched.remove(gi)
                break

    precision = (
        matched / len(preds)
        if preds
        else 0.0
    )

    recall = (
        matched / len(groups)
        if groups
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "list_precision": precision,
        "list_recall": recall,
        "list_f1": f1,
    }

## 35. ROUGE

In [ ]:
rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def get_gold_ideal(value):
    if value is None:
        return ""

    if isinstance(value, list):
        return (
            str(value[0]).strip()
            if value
            else ""
        )

    return str(value).strip()

def score_ideal(
    pred,
    gold,
    parse_valid=True
):
    # Conservative handling of malformed output.
    if not parse_valid:
        return {
            "rouge1_f": 0.0,
            "rouge2_f": 0.0,
            "rougeL_f": 0.0,
        }

    pred = str(
        pred or ""
    ).strip()

    gold = get_gold_ideal(
        gold
    )

    if not pred or not gold:
        return {
            "rouge1_f": 0.0,
            "rouge2_f": 0.0,
            "rougeL_f": 0.0,
        }

    s = rouge.score(
        gold,
        pred
    )

    return {
        "rouge1_f": s[
            "rouge1"
        ].fmeasure,
        "rouge2_f": s[
            "rouge2"
        ].fmeasure,
        "rougeL_f": s[
            "rougeL"
        ].fmeasure,
    }

## 36. Evaluate conditions without dropping invalid outputs

In [ ]:
gold_lookup = (
    val_df.set_index("id")[
        [
            "type",
            "exact_answer_gold",
            "ideal_answer_gold",
        ]
    ]
    .to_dict(orient="index")
)

def evaluate_results(results_df):
    scored_rows = []

    for row in results_df.itertuples(
        index=False
    ):
        qid = str(row.id)
        gold = gold_lookup[qid]
        qtype = row.type
        valid = bool(row.parse_valid)

        rec = {
            "id": qid,
            "type": qtype,
            "condition": row.condition,
            "parse_valid": valid,
            "yesno_correct": np.nan,
            "factoid_strict": np.nan,
            "factoid_lenient": np.nan,
            "factoid_mrr": np.nan,
            "list_precision": np.nan,
            "list_recall": np.nan,
            "list_f1": np.nan,
        }

        if qtype == "yesno":
            if valid:
                rec[
                    "yesno_correct"
                ] = float(
                    normalize_answer_text(
                        row.exact_answer_pred
                    )
                    == normalize_answer_text(
                        gold[
                            "exact_answer_gold"
                        ]
                    )
                )
            else:
                rec[
                    "yesno_correct"
                ] = 0.0

        elif qtype == "factoid":
            if valid:
                rec.update(
                    score_factoid(
                        row.exact_answer_pred,
                        gold[
                            "exact_answer_gold"
                        ]
                    )
                )
            else:
                rec.update({
                    "factoid_strict": 0.0,
                    "factoid_lenient": 0.0,
                    "factoid_mrr": 0.0,
                })

        elif qtype == "list":
            if valid:
                rec.update(
                    score_list(
                        row.exact_answer_pred,
                        gold[
                            "exact_answer_gold"
                        ]
                    )
                )
            else:
                rec.update({
                    "list_precision": 0.0,
                    "list_recall": 0.0,
                    "list_f1": 0.0,
                })

        rec.update(
            score_ideal(
                row.ideal_answer_pred,
                gold[
                    "ideal_answer_gold"
                ],
                parse_valid=valid
            )
        )

        scored_rows.append(rec)

    return pd.DataFrame(
        scored_rows
    )

## 37. Summarize answer quality

In [ ]:
def summarize_scored(
    scored_df,
    raw_results_df
):
    summaries = []

    for condition, group in (
        scored_df.groupby(
            "condition"
        )
    ):
        summary = {
            "condition": condition,
            "n": int(len(group)),
            "json_valid_rate": float(
                group[
                    "parse_valid"
                ].mean()
            ),
            "rouge1_f": float(
                group[
                    "rouge1_f"
                ].mean()
            ),
            "rouge2_f": float(
                group[
                    "rouge2_f"
                ].mean()
            ),
            "rougeL_f": float(
                group[
                    "rougeL_f"
                ].mean()
            ),
        }

        yes = group[
            group["type"] == "yesno"
        ]

        summary[
            "yesno_accuracy"
        ] = float(
            yes[
                "yesno_correct"
            ].mean()
        )

        y_true = []
        y_pred = []

        for qid in yes["id"]:
            gold_label = (
                normalize_answer_text(
                    gold_lookup[
                        str(qid)
                    ][
                        "exact_answer_gold"
                    ]
                )
            )

            r = raw_results_df[
                (
                    raw_results_df[
                        "condition"
                    ] == condition
                )
                & (
                    raw_results_df[
                        "id"
                    ].astype(str)
                    == str(qid)
                )
            ].iloc[0]

            pred_label = (
                normalize_answer_text(
                    r[
                        "exact_answer_pred"
                    ]
                )
                if bool(
                    r["parse_valid"]
                )
                else "__invalid__"
            )

            y_true.append(
                gold_label
            )
            y_pred.append(
                pred_label
            )

        summary[
            "yesno_macro_f1"
        ] = float(
            f1_score(
                y_true,
                y_pred,
                labels=[
                    "yes",
                    "no"
                ],
                average="macro",
                zero_division=0
            )
        )

        fact = group[
            group["type"] == "factoid"
        ]

        summary[
            "factoid_strict"
        ] = float(
            fact[
                "factoid_strict"
            ].mean()
        )
        summary[
            "factoid_lenient"
        ] = float(
            fact[
                "factoid_lenient"
            ].mean()
        )
        summary[
            "factoid_mrr"
        ] = float(
            fact[
                "factoid_mrr"
            ].mean()
        )

        lst = group[
            group["type"] == "list"
        ]

        summary[
            "list_precision"
        ] = float(
            lst[
                "list_precision"
            ].mean()
        )
        summary[
            "list_recall"
        ] = float(
            lst[
                "list_recall"
            ].mean()
        )
        summary[
            "list_f1"
        ] = float(
            lst[
                "list_f1"
            ].mean()
        )

        summary_q = group[
            group["type"] == "summary"
        ]

        summary[
            "summary_rougeL_f"
        ] = float(
            summary_q[
                "rougeL_f"
            ].mean()
        )

        summary[
            "macro_task_score"
        ] = float(
            np.mean([
                summary[
                    "yesno_accuracy"
                ],
                summary[
                    "factoid_mrr"
                ],
                summary[
                    "list_f1"
                ],
                summary[
                    "summary_rougeL_f"
                ],
            ])
        )

        summaries.append(
            summary
        )

    return pd.DataFrame(
        summaries
    )

In [ ]:
RUN_FULL_GENERATION = True
print("Full Phase 4B generation enabled:", RUN_FULL_GENERATION)

# Part H — Full Phase 4B generation

After the smoke test is accepted:

```python
RUN_FULL_GENERATION = True
```

New responses:

`862 × 2 = 1,724`

Phase 4A retrieved-RAG responses are reused; they are not regenerated.

## 38. Run full generation

In [ ]:
filtered_full = None

if RUN_FULL_GENERATION:
    filtered_full = generate_conditions(
        val_df,
        "validation_862q_2filtered_conditions"
    )

    assert len(
        filtered_full
    ) == 862 * 2

    filtered_full.to_csv(
        TABLE_DIR
        / "validation_862q_2filtered_conditions.csv",
        index=False
    )

    print(
        "New Phase 4B generations:",
        len(filtered_full)
    )
else:
    print(
        "RUN_FULL_GENERATION=False — "
        "full Phase 4B generation not started."
    )

## 39. Combine with Phase 4A retrieved-RAG baseline

In [ ]:
all_results = None

if filtered_full is not None:
    # Normalize columns expected by the evaluator.
    needed_cols = [
        "id",
        "type",
        "condition",
        "question",
        "evidence_tokens",
        "raw_output",
        "parse_valid",
        "parse_mode",
        "validation_status",
        "exact_answer_pred",
        "ideal_answer_pred",
    ]

    base = baseline_rag[
        needed_cols
    ].copy()

    new = filtered_full[
        needed_cols
    ].copy()

    all_results = pd.concat(
        [base, new],
        ignore_index=True
    )

    assert len(
        all_results
    ) == 862 * 3

    all_results.to_csv(
        TABLE_DIR
        / "phase4B_all_three_conditions.csv",
        index=False
    )

    display(
        all_results.groupby(
            ["condition", "type"]
        )["parse_valid"]
        .agg(["count", "mean"])
    )

## 40. Evaluate all three evidence conditions

In [ ]:
all_scored = None
summary_df = None

if all_results is not None:
    all_scored = evaluate_results(
        all_results
    )

    summary_df = summarize_scored(
        all_scored,
        all_results
    )

    all_scored.to_csv(
        TABLE_DIR
        / "phase4B_answer_scores_per_question.csv",
        index=False
    )

    summary_df.to_csv(
        TABLE_DIR
        / "phase4B_answer_summary.csv",
        index=False
    )

    display(
        summary_df.sort_values(
            "macro_task_score",
            ascending=False
        )
    )

## 41. ROUGE by condition and question type

In [ ]:
if all_scored is not None:
    rouge_by_type = (
        all_scored.groupby(
            ["condition", "type"]
        )
        .agg(
            n=("id", "size"),
            rouge1_f=(
                "rouge1_f",
                "mean"
            ),
            rouge2_f=(
                "rouge2_f",
                "mean"
            ),
            rougeL_f=(
                "rougeL_f",
                "mean"
            ),
        )
        .reset_index()
    )

    rouge_by_type.to_csv(
        TABLE_DIR
        / "rouge_by_condition_and_question_type.csv",
        index=False
    )

    display(
        rouge_by_type
    )

# Part I — Paired bootstrap comparisons

Primary comparisons:

- `document_top5 − retrieved_rag`
- `sentence_filtered − retrieved_rag`
- `sentence_filtered − document_top5`

Invalid outputs are already scored as zero and remain in all denominators.

## 42. Paired bootstrap helper

In [ ]:
def paired_bootstrap(
    scored_df,
    condition_a,
    condition_b,
    metric,
    qtype=None,
    n_boot=5000,
    seed=SEED
):
    df = scored_df.copy()

    if qtype is not None:
        df = df[
            df["type"] == qtype
        ]

    a = (
        df[
            df["condition"]
            == condition_a
        ][["id", metric]]
        .rename(
            columns={
                metric: "a"
            }
        )
    )

    b = (
        df[
            df["condition"]
            == condition_b
        ][["id", metric]]
        .rename(
            columns={
                metric: "b"
            }
        )
    )

    merged = a.merge(
        b,
        on="id",
        validate="one_to_one"
    )

    va = merged["a"].to_numpy(
        float
    )
    vb = merged["b"].to_numpy(
        float
    )

    rng = np.random.default_rng(
        seed
    )

    n = len(merged)

    deltas = []

    for _ in range(n_boot):
        idx = rng.integers(
            0,
            n,
            size=n
        )

        deltas.append(
            np.mean(
                va[idx]
            )
            - np.mean(
                vb[idx]
            )
        )

    deltas = np.asarray(
        deltas
    )

    return {
        "condition_a": condition_a,
        "condition_b": condition_b,
        "metric": metric,
        "qtype": (
            qtype
            if qtype
            else "all"
        ),
        "n": int(n),
        "mean_delta": float(
            deltas.mean()
        ),
        "ci95_low": float(
            np.quantile(
                deltas,
                0.025
            )
        ),
        "ci95_high": float(
            np.quantile(
                deltas,
                0.975
            )
        ),
        "bootstrap_probability_delta_gt_0": float(
            np.mean(
                deltas > 0
            )
        ),
    }

## 43. Bootstrap all key metrics

In [ ]:
if all_scored is not None:
    comparisons = [
        (
            "document_top5",
            "retrieved_rag"
        ),
        (
            "sentence_filtered",
            "retrieved_rag"
        ),
        (
            "sentence_filtered",
            "document_top5"
        ),
    ]

    specs = [
        (
            "rougeL_f",
            None
        ),
        (
            "yesno_correct",
            "yesno"
        ),
        (
            "factoid_strict",
            "factoid"
        ),
        (
            "factoid_lenient",
            "factoid"
        ),
        (
            "factoid_mrr",
            "factoid"
        ),
        (
            "list_f1",
            "list"
        ),
        (
            "rougeL_f",
            "summary"
        ),
    ]

    boot_rows = []

    for a, b in comparisons:
        for metric, qtype in specs:
            boot_rows.append(
                paired_bootstrap(
                    all_scored,
                    a,
                    b,
                    metric,
                    qtype=qtype
                )
            )

    bootstrap_df = pd.DataFrame(
        boot_rows
    )

    bootstrap_df.to_csv(
        TABLE_DIR
        / "paired_bootstrap_evidence_filtering.csv",
        index=False
    )

    display(
        bootstrap_df
    )

# Part J — Efficiency and trade-off analysis

## 44. Average evidence-token cost

In [ ]:
if all_results is not None:
    token_summary = (
        all_results.groupby(
            "condition"
        )["evidence_tokens"]
        .agg([
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max",
        ])
        .round(1)
        .reset_index()
    )

    token_summary.to_csv(
        TABLE_DIR
        / "evidence_token_cost_summary.csv",
        index=False
    )

    display(
        token_summary
    )

## 45. MacroTaskScore vs evidence-token cost figure

In [ ]:
if (
    summary_df is not None
    and all_results is not None
):
    token_means = (
        all_results.groupby(
            "condition"
        )["evidence_tokens"]
        .mean()
        .rename(
            "mean_evidence_tokens"
        )
        .reset_index()
    )

    efficiency_df = (
        summary_df.merge(
            token_means,
            on="condition",
            validate="one_to_one"
        )
    )

    efficiency_df.to_csv(
        TABLE_DIR
        / "evidence_efficiency_summary.csv",
        index=False
    )

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    for row in efficiency_df.itertuples(
        index=False
    ):
        ax.scatter(
            row.mean_evidence_tokens,
            row.macro_task_score
        )

        ax.annotate(
            row.condition,
            (
                row.mean_evidence_tokens,
                row.macro_task_score
            ),
            xytext=(5, 5),
            textcoords="offset points"
        )

    ax.set_xlabel(
        "Mean evidence tokens"
    )
    ax.set_ylabel(
        "MacroTaskScore"
    )
    ax.set_title(
        "Answer quality vs evidence context size"
    )

    fig.tight_layout()

    fig.savefig(
        FIG_DIR
        / "fig01_quality_vs_evidence_tokens.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

## 46. Main answer metrics figure

In [ ]:
if summary_df is not None:
    metric_df = (
        summary_df[
            [
                "condition",
                "yesno_accuracy",
                "factoid_mrr",
                "list_f1",
                "summary_rougeL_f",
                "macro_task_score",
            ]
        ]
        .set_index(
            "condition"
        )
    )

    fig, ax = plt.subplots(
        figsize=(9, 5.5)
    )

    metric_df.plot(
        kind="bar",
        ax=ax
    )

    ax.set_xlabel(
        "Evidence condition"
    )
    ax.set_ylabel(
        "Score"
    )
    ax.set_title(
        "Evidence-filtering validation performance"
    )
    ax.set_ylim(
        0,
        1
    )
    ax.tick_params(
        axis="x",
        rotation=0
    )

    fig.tight_layout()

    fig.savefig(
        FIG_DIR
        / "fig02_evidence_filtering_metrics.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

# Part K — Per-question improvement/degradation

## 47. Compare filtered systems with Top-10 RAG

In [ ]:
if all_scored is not None:
    type_metric = {
        "yesno": "yesno_correct",
        "factoid": "factoid_mrr",
        "list": "list_f1",
        "summary": "rougeL_f",
    }

    comparison_rows = []

    for qtype, metric in type_metric.items():
        subset = all_scored[
            all_scored["type"] == qtype
        ]

        pivot = subset.pivot(
            index="id",
            columns="condition",
            values=metric
        )

        for qid, r in pivot.iterrows():
            comparison_rows.append({
                "id": qid,
                "type": qtype,
                "metric": metric,
                "retrieved_rag": r[
                    "retrieved_rag"
                ],
                "document_top5": r[
                    "document_top5"
                ],
                "sentence_filtered": r[
                    "sentence_filtered"
                ],
            })

    change_df = pd.DataFrame(
        comparison_rows
    )

    change_df[
        "top5_minus_rag"
    ] = (
        change_df[
            "document_top5"
        ]
        - change_df[
            "retrieved_rag"
        ]
    )

    change_df[
        "sentence_minus_rag"
    ] = (
        change_df[
            "sentence_filtered"
        ]
        - change_df[
            "retrieved_rag"
        ]
    )

    body_lookup = (
        val_df.set_index(
            "id"
        )["body"]
        .to_dict()
    )

    change_df[
        "question"
    ] = (
        change_df[
            "id"
        ].map(
            body_lookup
        )
    )

    change_df.to_csv(
        TABLE_DIR
        / "per_question_filtering_changes.csv",
        index=False
    )

    print(
        "Largest sentence-filter improvements:"
    )

    display(
        change_df.sort_values(
            "sentence_minus_rag",
            ascending=False
        ).head(25)
    )

    print(
        "\nLargest sentence-filter degradations:"
    )

    display(
        change_df.sort_values(
            "sentence_minus_rag",
            ascending=True
        ).head(25)
    )

# Part L — Freeze the evidence strategy

Selection rule was fixed before full Phase 4B generation:

1. highest validation `MacroTaskScore`;
2. tie-break: higher overall `ROUGE-L`.

The selected evidence strategy is frozen for the next grounding/confidence phases.

If filtering does not beat standard Top-10 RAG, `retrieved_rag` remains the frozen evidence strategy.

## 48. Select frozen evidence strategy

In [ ]:
selection = None

if summary_df is not None:
    ranked = (
        summary_df.sort_values(
            [
                "macro_task_score",
                "rougeL_f",
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )

    best = ranked.iloc[0]

    selection = {
        "selection_rule": (
            "Highest MacroTaskScore; "
            "ROUGE-L tie-break"
        ),
        "macro_task_score_definition": (
            "mean(yesno_accuracy, "
            "factoid_mrr, list_f1, "
            "summary_rougeL_f)"
        ),
        "frozen_evidence_strategy": (
            best["condition"]
        ),
        "macro_task_score": float(
            best[
                "macro_task_score"
            ]
        ),
        "overall_rougeL_f": float(
            best[
                "rougeL_f"
            ]
        ),
        "internal_test_evaluated": False,
    }

    with open(
        PHASE4B_DIR
        / "frozen_evidence_strategy.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            selection,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        json.dumps(
            selection,
            indent=2,
            ensure_ascii=False
        )
    )

## 49. Phase 4B report

In [ ]:
if summary_df is not None:
    phase4b_report = {
        "dataset": "BioASQ14b",
        "validation_questions": 862,
        "internal_test_evaluated": False,
        "frozen_retrieval_pipeline": (
            frozen_selection
        ),
        "answer_generator": {
            "model": LLM_MODEL,
            "quantization": (
                "4bit_NF4"
                if USE_4BIT
                else "none"
            ),
            "decoding": "greedy",
            "do_sample": False,
        },
        "evidence_conditions": {
            "retrieved_rag": {
                "description": (
                    "Frozen reranked Top-10 "
                    "full title+abstract baseline"
                )
            },
            "document_top5": {
                "description": (
                    "First 5 documents from "
                    "the frozen reranked list"
                ),
                "top_k": DOCUMENT_TOP_K,
            },
            "sentence_filtered": {
                "description": (
                    "Sentence-level zero-shot "
                    "MedCPT Cross-Encoder filtering"
                ),
                "source_top_k": SOURCE_TOP_K,
                "max_sentences_per_document": (
                    SENTENCE_MAX_PER_DOCUMENT
                ),
                "max_sentences_total": (
                    SENTENCE_MAX_TOTAL
                ),
                "token_budget": (
                    SENTENCE_FILTER_TOKEN_BUDGET
                ),
                "score_use": (
                    "ranking only; raw logits "
                    "not interpreted as calibrated probabilities"
                ),
            },
        },
        "automatic_validation_results": (
            summary_df.to_dict(
                orient="records"
            )
        ),
        "selection": selection,
    }

    with open(
        PHASE4B_DIR
        / "phase4B_report.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            phase4b_report,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        json.dumps(
            phase4b_report,
            indent=2,
            ensure_ascii=False
        )
    )

# STOP POINT 4B-FULL

After the full run, send:

1. evidence-length comparison
2. gold-document coverage after filtering
3. JSON compliance table
4. `phase4B_answer_summary`
5. `rouge_by_condition_and_question_type`
6. paired-bootstrap evidence-filtering table
7. quality-vs-evidence-token figure
8. main metrics figure
9. largest sentence-filter improvements/degradations
10. `frozen_evidence_strategy.json`
11. `phase4B_report.json`

Then Phase 4C will evaluate **evidence grounding / faithfulness** using the frozen evidence strategy.

Do not evaluate the internal test.

## 50. Final safety check

In [ ]:
if not ALLOW_INTERNAL_TEST_EVALUATION:
    print("INTERNAL TEST REMAINS LOCKED.")
else:
    raise RuntimeError(
        "Do not unlock the internal test during Phase 4B."
    )

print("Phase 4B output:", PHASE4B_DIR)